# Feature Engineering

## Objective

The objective of this phase is to transform the cleaned dataset into a format suitable for machine learning models.

This includes selecting relevant features, separating predictors from the target, encoding categorical variables, scaling numerical variables when appropriate, and preparing a reproducible preprocessing pipeline.

In [1]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
# Load cleaned dataset

df = pd.read_csv("../data/processed/customer_churn_clean.csv")

display(df.head())

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Feature Selection

The `customerID` variable is an identifier rather than a meaningful predictive feature.

It will be removed because customer identifiers do not provide useful information about churn behavior and could introduce noise into the model.

In [3]:
# Remove identifier column

df = df.drop(columns=["customerID"])

## Separating Features and Target

The target variable represents the outcome that the model will predict.

All remaining variables will be used as potential predictors.

In [4]:
# Separate features and target

X = df.drop(columns=["Churn"])
y = df["Churn"]

## Encoding the Target Variable

The `Churn` target is categorical, containing `Yes` and `No`.

Machine learning classifiers require a numerical representation of the target, so the values will be encoded as 1 for churn and 0 for non-churn.

In [5]:
# Encode target variable

y = y.map({"No": 0, "Yes": 1})

display(y.value_counts())

Churn
0    5163
1    1869
Name: count, dtype: int64

## Train/Test Split

The dataset is divided into training and testing sets before fitting any preprocessing transformations.

The training set will be used to learn patterns and preprocessing parameters, while the test set will remain unseen until final model evaluation.

Stratification is used to preserve the proportion of churned and non-churned customers in both subsets.

In [6]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (5625, 19)
Testing set: (1407, 19)


In [7]:
# Define categorical and numerical features

categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

## Preprocessing Pipeline

Categorical features will be transformed using One-Hot Encoding so that machine learning algorithms can work with them numerically.

Numerical features will be standardized using StandardScaler.

Using a preprocessing pipeline prevents data leakage because transformations are fitted only on the training data.

In [8]:
# Create preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [9]:
# Fit preprocessing pipeline on training data

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

## Processed Feature Verification

The preprocessing pipeline converts the original mixed-type dataset into a numerical feature matrix suitable for machine learning algorithms.

In [10]:
# Display processed dataset dimensions

print("Processed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)

Processed training data shape: (5625, 46)
Processed testing data shape: (1407, 46)


In [11]:
# Retrieve generated feature names

feature_names = preprocessor.get_feature_names_out()

display(feature_names)

array(['numerical__tenure', 'numerical__MonthlyCharges',
       'numerical__TotalCharges', 'categorical__gender_Female',
       'categorical__gender_Male', 'categorical__SeniorCitizen_0',
       'categorical__SeniorCitizen_1', 'categorical__Partner_No',
       'categorical__Partner_Yes', 'categorical__Dependents_No',
       'categorical__Dependents_Yes', 'categorical__PhoneService_No',
       'categorical__PhoneService_Yes', 'categorical__MultipleLines_No',
       'categorical__MultipleLines_No phone service',
       'categorical__MultipleLines_Yes',
       'categorical__InternetService_DSL',
       'categorical__InternetService_Fiber optic',
       'categorical__InternetService_No',
       'categorical__OnlineSecurity_No',
       'categorical__OnlineSecurity_No internet service',
       'categorical__OnlineSecurity_Yes', 'categorical__OnlineBackup_No',
       'categorical__OnlineBackup_No internet service',
       'categorical__OnlineBackup_Yes',
       'categorical__DeviceProtection_

In [12]:
# Verify training and testing target distributions

print("Training churn rate:")
display(y_train.mean().round(4))

print("\nTesting churn rate:")
display(y_test.mean().round(4))

Training churn rate:


np.float64(0.2658)


Testing churn rate:


np.float64(0.2658)

## Final Observations

- The `customerID` identifier was removed because it does not represent a meaningful predictive feature.
- The target variable was encoded as a binary variable.
- The dataset was divided into training and testing subsets using stratification.
- Categorical features were transformed using One-Hot Encoding.
- Numerical features were standardized using StandardScaler.
- Preprocessing was fitted exclusively on the training data to prevent data leakage.
- The resulting feature matrix is ready for machine learning model training.